# Load design into a canonical Model

This notebook separates transport, interpretation, validation, and presentation. Fixture mode is a small synthetic public design. Live mode reads the declared pinned service object; it never publishes. An error does not switch modes. Geometry stays in its source transport.

In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display, IFrame
from rangekeeper.examples import design
from rangekeeper.model import Model
from rangekeeper.graph import View, Hierarchy
from rangekeeper.graph.projection import to_table, FieldColumn, ValueColumn, PropertyColumn
from rangekeeper.adapters import cytoscape
from rangekeeper.adapters.plotting import plot_flows, plot_partition
from rangekeeper.io import json as codec

MODE = os.environ.get("RK_DESIGN_MODE", "fixture")
print("Declared design mode:", MODE)
OUTPUT = Path("design-output")
OUTPUT.mkdir(exist_ok=True)


## Receive or load

Historical Speckle content requires an explicit reviewed mapping. A canonical envelope can instead use `decode_model`. Collection nesting does not establish domain membership.

In [ ]:
if MODE == "fixture":
    source = design.fixture()
    utility_names = {"cores": "cores"}
elif MODE == "live":
    from specklepy.api.client import SpeckleClient
    from rangekeeper.adapters.speckle import receive
    from rangekeeper.migration.speckle import MappingSpec, convert_speckle
    client = SpeckleClient(host="app.speckle.systems")
    client.authenticate_with_token(os.environ["SPECKLE_TOKEN"])
    received = receive(client, project_id="c0f66c35e3", object_id="e7acaac21ae7e9369339900a4aaeb827")
    mapping = MappingSpec(
        classifications=("floor", "space", "utilities", "building", "property"),
        relationships=("spatiallyContains", "services", "contains"),
        measurements={"gfa": "meter ** 2", "volume": "meter ** 3", "ffl": "meter", "ftf": "meter", "perimeter": "meter"},
        properties=("use", "number"),
        source_name="Speckle c0f66c35e3/e7acaac21ae7e9369339900a4aaeb827",
    )
    conversion = convert_speckle(received.payload, mapping=mapping)
    assert conversion.model is not None, conversion.issues
    source = conversion.model
    utility_names = {"buildingAcores": "cores", "buildingBcores": "cores", "plinthplant": "plant"}
elif MODE == "canonical":
    from rangekeeper.adapters.speckle import decode_model
    import json
    source = decode_model(json.loads(Path(os.environ["RK_DESIGN_ENVELOPE"]).read_text()))
    utility_names = json.loads(Path(os.environ["RK_DESIGN_INTERPRETATION"]).read_text())["utility_names"]
else:
    raise ValueError("Select fixture, live, or canonical explicitly")
source = Model.from_data(source.to_data())
print("Model revision:", source.id, "Objects:", len(source.find_entities()))


## Select by identity

Classification codes are resolved once to UUIDs. Names remain search labels. Spatial containment is selected separately from services and other relationships. This tree projection checks for multiple parents; the ordinary graph viewer also supports shared membership.

In [ ]:
spatial = design.classification_id(source, "spatiallyContains")
spatial_view = View(source, relationships=[e.id for e in source.system.relationships if e.classification == spatial])
hierarchy = Hierarchy(spatial_view)
root = hierarchy.root
contributors = design.select_contributors(source, root=root)
table = to_table(spatial_view, columns=(
    FieldColumn("ID", "entity_id"), FieldColumn("Name", "name"),
    FieldColumn("Class", "classification_code"), ValueColumn("GFA (m²)", "gfa", "meter ** 2"),
    PropertyColumn("Use", "use"), PropertyColumn("Number", "number"),
))
display(pd.DataFrame([dict(row.values) for row in table.rows]))
viewer = cytoscape.write_viewer([cytoscape.project(spatial_view, "Spatial containment")], OUTPUT / "spatial.html")
display(IFrame(src=str(viewer), width="100%", height=600))


## Report detached data

Areas below use unique floor identities. Parent totals are calculated for presentation and are not added to their constituents.

In [ ]:
from rangekeeper.model.characteristics import value as local_value
from rangekeeper.units import default_units
areas = {}
for uid in contributors:
    item = local_value(source.entity(uid).characteristics, "gfa")
    if item is not None:
        assert item.quantity is not None, f"Unresolved GFA: {uid}"
        areas[uid] = default_units.convert(item.quantity, to="meter ** 2").magnitude
for chart in ("sunburst", "treemap"):
    figure = plot_partition(hierarchy, areas, units="m²", title="Gross floor area", kind=chart)
    figure.write_html(OUTPUT / f"gfa-{chart}.html", include_plotlyjs=True)
    display(figure)
